# Engine Freshness Index (EFI): Research Methodology

This notebook implements the methodology used to construct and evaluate the **Engine Freshness Index (EFI)**, a continuous multivariate score derived from petrol-vehicle emission-inspection measurements.

## Workflow

1. Load January and February records as the development dataset.
2. Apply vehicle-type, completeness, RPM, and compliance-consistency filtering.
3. Standardize 12 acceleration and idle mode emission/operating variables.
4. Use K-means clustering to identify three multivariate emission profiles.
5. Select the cleanest empirical cluster and use its centroid as the EFI reference state.
6. Calculate Euclidean distance from the clean centroid.
7. Empirically calibrate the distance scale and invert it to obtain EFI.
8. Train regression models to approximate the derived EFI directly from raw measurements.
9. Interpret the selected Random Forest Lite model using feature importance and SHAP.
10. Evaluate model behaviour on the independent March dataset.

> **Data availability:** The raw vehicle-inspection records are not distributed in this repository. Authorized copies of the monthly CSV files must be placed in the local `data/` directory before execution.


## 1. Imports and reproducibility configuration


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ks_2samp
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import shap

RANDOM_STATE = 42

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
})

## 2. Data configuration

January and February are combined to form the development dataset. March is kept separate for temporal evaluation.

Expected local files:

- `data/January2025.csv`
- `data/February2025.csv`
- `data/March2025.csv`


In [ ]:
DATA_DIR = Path("../data")
FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

JANUARY_FILE = DATA_DIR / "January2025.csv"
FEBRUARY_FILE = DATA_DIR / "February2025.csv"
MARCH_FILE = DATA_DIR / "March2025.csv"

FEATURE_COLUMNS = [
    "AccHC", "AccCO", "AccCO2", "AccO2", "AccLambda", "AccRPM",
    "IdleHC", "IdleCO", "IdleCO2", "IdleO2", "IdleLambda", "IdleRPM",
]

HC_THRESHOLD = 6000
CO_THRESHOLD = 4

In [ ]:
def load_required_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Required data file not found: {path}\n"
            "Place the authorized dataset in the repository's data/ directory."
        )
    return pd.read_csv(path)

january_raw = load_required_csv(JANUARY_FILE)
february_raw = load_required_csv(FEBRUARY_FILE)
march_raw = load_required_csv(MARCH_FILE)

development_raw = pd.concat(
    [january_raw, february_raw],
    ignore_index=True,
)

print(f"January rows: {len(january_raw):,}")
print(f"February rows: {len(february_raw):,}")
print(f"March rows: {len(march_raw):,}")

## 3. Feature selection and preprocessing

### 3.1 Feature selection rationale

EFI is constructed from the 12 continuous emission and operating measurements available for petrol-powered motor cars in the vehicle-emission inspection dataset:

- HC, CO, CO₂, O₂, lambda, and RPM under acceleration conditions; and
- HC, CO, CO₂, O₂, lambda, and RPM under idle conditions.

These variables were selected because they are the continuous measurements collected by the vehicle-emission testing programme for petrol-powered motor cars and were therefore the available quantitative emission and operating features for this vehicle category.

Using measurements from both acceleration and idle test conditions allows the EFI formulation to characterize the multivariate emission state across both operating conditions represented in the inspection procedure, rather than relying on a single pollutant or operating condition.


### 3.2 Preprocessing rules

The analysis retains petrol-powered motor cars and removes records that cannot support the EFI formulation. The preprocessing steps are:

- retain petrol vehicles;
- retain vehicle classes containing `motor car`;
- remove records with `TestResult == 'A'`;
- apply the compliance-consistency rule described below;
- retain the 12 EFI variables and `TestResult`;
- remove records with missing EFI variables; and
- require positive acceleration and idle RPM values.


### 3.3 Compliance-based consistency filtering

The dataset contains a binary inspection result (`TestResult`) in addition to the measured emission values. Failing records are checked for consistency with the applicable petrol-vehicle emission compliance thresholds used by the government vehicle-emission testing programme.

For this consistency check, the relevant thresholds are:
- HC < 6000
- CO < 4

under both acceleration and idle test conditions.

A record marked as `Fail` while simultaneously satisfying all four threshold conditions is treated as inconsistent with the corresponding measured compliance values:

- Acceleration HC < 6000
- Acceleration CO < 4
- Idle HC < 6000
- Idle CO < 4

Such records are excluded before EFI formulation. This is a result consistency rule based on the inspection thresholds, rather than an ML based outlier removal procedure.


In [ ]:
def preprocess_emission_data(df: pd.DataFrame):
    required_columns = [
        "VehFuelType",
        "VehClass",
        "TestResult",
        *FEATURE_COLUMNS,
    ]

    missing_columns = [
        col for col in required_columns if col not in df.columns
    ]
    if missing_columns:
        raise KeyError(f"Missing required columns: {missing_columns}")

    work = df.loc[
        (df["VehFuelType"].astype(str).str.lower() == "petrol")
        & (df["VehClass"].astype(str).str.lower().str.contains("motor car", na=False))
        & (df["TestResult"].astype(str).str.lower() != "a")
    ].copy()

    excluded_failures_mask = (
        (work["TestResult"] == "F")
        & (work["AccHC"] < HC_THRESHOLD)
        & (work["AccCO"] < CO_THRESHOLD)
        & (work["IdleHC"] < HC_THRESHOLD)
        & (work["IdleCO"] < CO_THRESHOLD)
    )

    excluded_failure_count = int(excluded_failures_mask.sum())

    work = work.loc[
        ~excluded_failures_mask,
        FEATURE_COLUMNS + ["TestResult"],
    ].copy()

    work = work.dropna()
    work = work.loc[(work[["AccRPM", "IdleRPM"]] > 0).all(axis=1)]
    work = work.reset_index(drop=True)

    return work, excluded_failure_count

In [ ]:
development_df, development_excluded_count = preprocess_emission_data(
    development_raw
)
march_df, march_excluded_count = preprocess_emission_data(
    march_raw
)

print(f"Development records after preprocessing: {len(development_df):,}")
print(
    "Compliance-inconsistent failure records excluded "
    f"(development): {development_excluded_count:,}"
)
print(f"March records after preprocessing: {len(march_df):,}")
print(
    "Compliance-inconsistent failure records excluded "
    f"(March): {march_excluded_count:,}"
)

### 3.4 Standardization

The EFI variables are measured on different numerical scales and units. Both K-means clustering and the EFI formulation rely on Euclidean distance. Without standardization, variables with larger numerical magnitudes could disproportionately dominate those distances.

`StandardScaler` is therefore fitted to the January–February development data and used to transform the 12 variables into a common standardized feature space before clustering and centroid-distance calculation.


In [ ]:
efi_scaler = StandardScaler()
X_development_scaled = efi_scaler.fit_transform(
    development_df[FEATURE_COLUMNS]
)

print(
    "Standardized development matrix shape:",
    X_development_scaled.shape,
)

## 4. Unsupervised emission-profile discovery

Candidate K-means solutions from $k=2$ to $k=9$ are evaluated using:

- within-cluster sum of squares (SSE), to examine the reduction in within-cluster dispersion and,
- silhouette score, to examine separation and cohesion of the resulting clusters.


In [ ]:
candidate_k = range(2, 10)
sse = []
silhouette_scores = []

for k in candidate_k:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10,
    )
    labels = model.fit_predict(X_development_scaled)

    sse.append(model.inertia_)
    silhouette_scores.append(
        silhouette_score(X_development_scaled, labels)
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(list(candidate_k), sse, marker="o")
axes[0].set_title("Elbow Method (SSE)")
axes[0].set_xlabel("k")
axes[0].set_ylabel("SSE")

axes[1].plot(list(candidate_k), silhouette_scores, marker="s")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Score")

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "kmeans_selection.png",
    bbox_inches="tight",
)
plt.show()

### 4.1 Selection of three clusters

The SSE curve shows its strongest reduction at the lower values of $k$, with progressively smaller reductions as additional clusters are introduced. Silhouette scores are also substantially stronger for $k=2$, $k=3$, and $k=4$ than for solutions with five or more clusters.

The highest silhouette score is not uniquely associated with $k=3$. The three-cluster solution is retained because it combines strong separation with a practically interpretable representation of three broad engine-emission profiles, while avoiding unnecessary division of those profiles.

Therefore, $k=3$ is used for subsequent EFI development as a balance between statistical separation and domain interpretability.


In [ ]:
kmeans_3 = KMeans(
    n_clusters=3,
    random_state=RANDOM_STATE,
    n_init=10,
)

development_df["Cluster_k3"] = kmeans_3.fit_predict(
    X_development_scaled
)

cluster_counts = (
    development_df["Cluster_k3"]
    .value_counts()
    .sort_index()
)

print("Cluster sizes:")
print(cluster_counts)

## 5. Characterizing the three emission profiles


In [ ]:
cluster_means = (
    development_df
    .groupby("Cluster_k3")[FEATURE_COLUMNS]
    .mean()
)

cluster_means

### 5.1 CO₂ behaviour by inspection result

As part of the cluster interpretation, acceleration and idle CO₂ measurements are examined against the recorded inspection result. This plot shows how the two CO₂ measurements are distributed among passing and failing vehicles in the development data.


In [ ]:
co2_plot_df = development_df[
    ["AccCO2", "IdleCO2", "TestResult"]
].copy()

co2_plot_df["Inspection_Result"] = (
    co2_plot_df["TestResult"]
    .map({"P": "Pass", "F": "Fail"})
    .fillna(co2_plot_df["TestResult"].astype(str))
)

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=co2_plot_df,
    x="AccCO2",
    y="IdleCO2",
    hue="Inspection_Result",
    alpha=0.45,
    s=28,
)

for result_label in ["Pass", "Fail"]:
    subset = co2_plot_df.loc[
        co2_plot_df["Inspection_Result"] == result_label
    ]
    if len(subset) > 1:
        sns.regplot(
            data=subset,
            x="AccCO2",
            y="IdleCO2",
            scatter=False,
            ci=None,
            line_kws={"linestyle": "--"},
        )

plt.title(
    "Acceleration CO₂ vs Idle CO₂ by Inspection Result"
)
plt.xlabel("Acceleration CO₂ (%)")
plt.ylabel("Idle CO₂ (%)")
plt.legend(title="Inspection Result")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "co2_acceleration_vs_idle_by_result.png",
    bbox_inches="tight",
)
plt.show()

### 5.2 Selection of the clean reference cluster

The EFI requires an empirical reference profile representing comparatively clean emission behaviour. The mean emission characteristics of the three K-means clusters are therefore compared under both acceleration and idle conditions.

Cluster 2 is distinguished by:
- the lowest mean HC under both acceleration and idle conditions;
- the lowest mean CO under both conditions;
- the lowest mean O₂ under both conditions; and
- the highest mean CO₂ under both conditions.

The acceleration versus idle CO₂ analysis also shows that passing vehicles are more strongly concentrated in the higher CO₂ region of the observed dataset than failing vehicles. Within the multivariate measurements available in this inspection dataset, the combined HC, CO, O₂, and CO₂ profile therefore identifies Cluster 2 as the cleanest of the three empirical emission profiles.

Cluster 2 is then used as the clean-reference cluster for EFI construction.

In [ ]:
selected_cluster_features = [
    "AccHC", "AccCO", "AccCO2", "AccO2", "AccLambda",
    "IdleHC", "IdleCO", "IdleCO2", "IdleO2", "IdleLambda",
]

cluster_profile = cluster_means[selected_cluster_features]

plt.figure(figsize=(12, 6))
sns.heatmap(
    cluster_profile.T,
    annot=True,
    fmt=".2f",
    cmap="viridis",
)

plt.title("Mean Emission Profile by K-means Cluster")
plt.xlabel("Cluster")
plt.ylabel("Emission variable")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "cluster_profiles.png",
    bbox_inches="tight",
)
plt.show()

## 6. Engine Freshness Index formulation

### 6.1 Distance from the clean reference state

After selecting the clean reference cluster, its centroid is used as a compact representation of the average clean-emission profile in the standardized 12-dimensional feature space.

Euclidean distance quantifies how far each vehicle's multivariate emission profile lies from this reference point. Because the variables have already been standardized, the distance combines deviations across the 12 measurements without allowing variables with larger original numerical scales to dominate solely because of their units.

For vehicle $i$, with standardized feature vector $z_i$ and clean-reference centroid $c_{\mathrm{clean}}$:

$$
d_i = \lVert z_i - c_{\mathrm{clean}} \rVert_2
$$

A smaller distance indicates greater similarity to the clean reference state, whereas a larger distance indicates greater multivariate deviation from that reference.


In [ ]:
CLEAN_CLUSTER_ID = 2

clean_cluster_mask = development_df["Cluster_k3"].eq(
    CLEAN_CLUSTER_ID
)

clean_cluster_points = X_development_scaled[
    clean_cluster_mask.to_numpy()
]

clean_centroid = clean_cluster_points.mean(axis=0)

distance_to_clean_centroid = np.linalg.norm(
    X_development_scaled - clean_centroid,
    axis=1,
)

print(f"Clean reference cluster: {CLEAN_CLUSTER_ID}")
print(f"Clean-cluster samples: {clean_cluster_mask.sum():,}")

### 6.2 Empirical calibration of the distance scale

The unscaled centroid-distance distribution is sorted and inspected before the final EFI orientation is applied.

The distribution contains a broad main body followed by a small extreme upper tail. During EFI development, the transition into this rapidly increasing tail was identified at approximately a distance derived score of **22**.

There are 138 observations beyond this point, corresponding to approximately 0.35% of the development dataset. The 0.35% proportion is therefore an empirical characteristic of the observed extreme tail. It was not imposed in advance as an assumed outlier rate.

This empirically observed tail proportion is used as the calibration target for selecting the distance reference of the nominal EFI scale.

In [ ]:
distance_sorted = np.sort(distance_to_clean_centroid)

plt.figure(figsize=(9, 5))
plt.plot(distance_sorted)
plt.axhline(
    22,
    linestyle="--",
    label="Exploratory tail transition ≈ 22",
)

plt.title(
    "Sorted Distance-Derived Scores Before EFI Inversion"
)
plt.xlabel("Vehicles (sorted by distance)")
plt.ylabel("Distance-derived score")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "sorted_distance_before_efi_inversion.png",
    bbox_inches="tight",
)
plt.show()

tail_threshold = 22.0
tail_count = int(
    (distance_to_clean_centroid > tail_threshold).sum()
)
tail_percentage = (
    tail_count / len(distance_to_clean_centroid)
) * 100

print(f"Observations above {tail_threshold:g}: {tail_count:,}")
print(f"Percentage of development data: {tail_percentage:.2f}%")

### 6.3 Selection of the distance reference

Several candidate distance-reference values are evaluated by scaling the centroid distances and measuring the proportion of observations that exceed the nominal upper boundary of 100 before score inversion.

The objective is to retain approximately the same 0.35% extreme-tail proportion identified in the unscaled distance distribution, while mapping the remaining approximately 99.65% of observations into the nominal scale.


In [ ]:
candidate_references = [22.0, 15.0, 12.0, 13.0]

calibration_results = []

for reference in candidate_references:
    scaled_distance = (
        distance_to_clean_centroid / reference
    ) * 100.0

    outside_count = int(
        (scaled_distance > 100).sum()
    )
    outside_percentage = (
        outside_count / len(scaled_distance)
    ) * 100

    calibration_results.append({
        "Distance reference": reference,
        "Observations > 100": outside_count,
        "Percentage": outside_percentage,
    })

calibration_results_df = pd.DataFrame(
    calibration_results
)

calibration_results_df

A distance reference of **13** is selected because it places approximately 0.35% of the development observations outside the nominal upper boundary before inversion, closely reproducing the empirically identified extreme tail proportion.

The value 13 is therefore an **empirically calibrated scale parameter**. It is not a compulsory emission threshold and does not represent a physical pollutant limit. Its role is to map the main body of the observed distance distribution onto an interpretable nominal scale while preserving the small extreme tail rather than compressing the complete distribution to accommodate those observations.


### 6.4 Score orientation

Centroid distance has the opposite direction from the intended interpretation of EFI. smaller distance represents greater similarity to the clean reference, while larger distance represents greater deviation.

For interpretability, the scaled distance is inverted:

$$
EFI_i = 100 - \left(\frac{d_i}{13}\right) \times 100
$$

The resulting orientation is:

- **higher EFI** -> greater similarity to the clean reference profile.
- **lower EFI** -> greater deviation from the clean reference profile.

An observation located exactly at the clean-cluster centroid receives an EFI of 100.

The score is not clipped at zero. Observations in the extreme distance tail can therefore receive negative EFI values. Retaining these values preserves information about the magnitude of extreme deviation instead of forcing all such observations to the same boundary value.


In [ ]:
EFI_DISTANCE_REFERENCE = 13.0

development_df["EFI_Score"] = (
    100.0
    - (
        distance_to_clean_centroid
        / EFI_DISTANCE_REFERENCE
    ) * 100.0
)

negative_efi_count = int(
    (development_df["EFI_Score"] < 0).sum()
)
negative_efi_pct = (
    100 * negative_efi_count / len(development_df)
)

print(
    f"EFI minimum: "
    f"{development_df['EFI_Score'].min():.4f}"
)
print(
    f"EFI maximum: "
    f"{development_df['EFI_Score'].max():.4f}"
)
print(
    f"EFI mean: "
    f"{development_df['EFI_Score'].mean():.4f}"
)
print(f"EFI < 0 records: {negative_efi_count:,}")
print(f"EFI < 0 percentage: {negative_efi_pct:.2f}%")

In [ ]:
efi_sorted = np.sort(
    development_df["EFI_Score"]
)

plt.figure(figsize=(9, 5))
plt.plot(efi_sorted)
plt.axhline(0, linestyle="--")

plt.title(
    "Sorted EFI Distribution Following Scaling and Inversion"
)
plt.xlabel("Vehicles (sorted by EFI)")
plt.ylabel("EFI")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "sorted_efi_after_scaling_inversion.png",
    bbox_inches="tight",
)
plt.show()

## 7. EFI characterization

The derived EFI is characterized using its overall distribution, its relationship with the K-means profiles and inspection result, and its correlations with the 12 input measurements.


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(
    development_df["EFI_Score"],
    bins=50,
    kde=True,
)

plt.title("EFI Score Distribution")
plt.xlabel("EFI Score")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "efi_distribution.png",
    bbox_inches="tight",
)
plt.show()

In [ ]:
cluster_efi_summary = (
    development_df
    .groupby("Cluster_k3")["EFI_Score"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

cluster_efi_summary

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=development_df,
    x="Cluster_k3",
    y="EFI_Score",
)

plt.title("EFI Score Distribution by K-means Cluster")
plt.xlabel("K-means cluster")
plt.ylabel("EFI Score")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "efi_by_cluster.png",
    bbox_inches="tight",
)
plt.show()

In [ ]:
pass_fail_summary = (
    development_df
    .groupby("TestResult")["EFI_Score"]
    .agg(["count", "mean", "median", "std"])
)

pass_fail_summary

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=development_df,
    x="TestResult",
    y="EFI_Score",
)

plt.title("EFI Score Distribution by Inspection Result")
plt.xlabel("Inspection result")
plt.ylabel("EFI Score")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "efi_by_test_result.png",
    bbox_inches="tight",
)
plt.show()

In [ ]:
efi_correlations = (
    development_df[
        FEATURE_COLUMNS + ["EFI_Score"]
    ]
    .corr()["EFI_Score"]
    .drop("EFI_Score")
    .sort_values()
)

efi_correlations.to_frame(
    "Correlation_with_EFI"
)

## 8. Supervised learning of EFI

EFI is a predictable score derived from the same 12 emission measurements used as model inputs. The supervised models below therefore learn a **mapping from raw emission measurements to the derived EFI**. They do not predict an independently measured engine-health ground truth variable.

The evaluated regressors are,
- Linear Regression
- Decision Tree
- Random Forest
- Random Forest Lite
- XGBoost
- LightGBM

The Random Forest Lite implementation uses a constrained configuration of 50 trees, maximum depth 12, and minimum leaf size 5.

In [ ]:
X_raw = development_df[
    FEATURE_COLUMNS
].to_numpy()

y = development_df[
    "EFI_Score"
].to_numpy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

linear_scaler = StandardScaler()
X_linear = linear_scaler.fit_transform(
    development_df[FEATURE_COLUMNS]
)

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_linear,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"Training samples: {len(y_train):,}")
print(
    "Held-out development test samples: "
    f"{len(y_test):,}"
)

In [ ]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(
        y_true,
        y_pred,
    )

    return {
        "R2": r2_score(y_true, y_pred),
        "MSE": mse,
        "MAE": mean_absolute_error(
            y_true,
            y_pred,
        ),
        "RMSE": np.sqrt(mse),
    }

### 8.1 Linear Regression


In [ ]:
linear_model = LinearRegression()
linear_model.fit(
    X_train_lr,
    y_train_lr,
)

linear_train_pred = linear_model.predict(
    X_train_lr
)
linear_test_pred = linear_model.predict(
    X_test_lr
)

### 8.2 Decision Tree


In [ ]:
decision_tree = DecisionTreeRegressor(
    random_state=RANDOM_STATE
)
decision_tree.fit(
    X_train_raw,
    y_train,
)

dt_train_pred = decision_tree.predict(
    X_train_raw
)
dt_test_pred = decision_tree.predict(
    X_test_raw
)


### 8.3 Random Forest


In [ ]:
random_forest = RandomForestRegressor(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
random_forest.fit(
    X_train_raw,
    y_train,
)

rf_train_pred = random_forest.predict(
    X_train_raw
)
rf_test_pred = random_forest.predict(
    X_test_raw
)

### 8.4 Random Forest Lite


In [ ]:
rf_lite = RandomForestRegressor(
    n_estimators=50,
    max_depth=12,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_lite.fit(
    X_train_raw,
    y_train,
)

rf_lite_train_pred = rf_lite.predict(
    X_train_raw
)
rf_lite_test_pred = rf_lite.predict(
    X_test_raw
)

### 8.5 XGBoost


In [ ]:
xgboost_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
)
xgboost_model.fit(
    X_train_raw,
    y_train,
)

xgb_train_pred = xgboost_model.predict(
    X_train_raw
)
xgb_test_pred = xgboost_model.predict(
    X_test_raw
)


### 8.6 LightGBM


In [ ]:
lightgbm_model = LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
)
lightgbm_model.fit(
    X_train_raw,
    y_train,
)

lgbm_train_pred = lightgbm_model.predict(
    X_train_raw
)
lgbm_test_pred = lightgbm_model.predict(
    X_test_raw
)

### 8.7 Model comparison


In [ ]:
model_results = []

model_predictions = {
    "Linear Regression": (
        y_train_lr,
        linear_train_pred,
        y_test_lr,
        linear_test_pred,
    ),
    "Decision Tree": (
        y_train,
        dt_train_pred,
        y_test,
        dt_test_pred,
    ),
    "Random Forest": (
        y_train,
        rf_train_pred,
        y_test,
        rf_test_pred,
    ),
    "Random Forest Lite": (
        y_train,
        rf_lite_train_pred,
        y_test,
        rf_lite_test_pred,
    ),
    "XGBoost": (
        y_train,
        xgb_train_pred,
        y_test,
        xgb_test_pred,
    ),
    "LightGBM": (
        y_train,
        lgbm_train_pred,
        y_test,
        lgbm_test_pred,
    ),
}

for model_name, (
    y_train_true,
    y_train_pred,
    y_test_true,
    y_test_pred,
) in model_predictions.items():

    train_metrics = regression_metrics(
        y_train_true,
        y_train_pred,
    )
    test_metrics = regression_metrics(
        y_test_true,
        y_test_pred,
    )

    model_results.append({
        "Model": model_name,
        "Train_R2": train_metrics["R2"],
        "Test_R2": test_metrics["R2"],
        "Test_MSE": test_metrics["MSE"],
        "Test_MAE": test_metrics["MAE"],
        "Test_RMSE": test_metrics["RMSE"],
    })

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        "Test_R2",
        ascending=False,
    )
)

model_results_df

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=model_results_df,
    x="Model",
    y="Test_R2",
)

plt.title(
    "Held-out R² by Regression Model"
)
plt.xlabel("")
plt.ylabel("Test R²")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "model_comparison_r2.png",
    bbox_inches="tight",
)
plt.show()

## 9. Random Forest Lite diagnostics

The following diagnostics examine agreement between derived and predicted EFI, residual behaviour, and learning-curve performance for the Random Forest Lite model.


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(
    y_test,
    rf_lite_test_pred,
    alpha=0.4,
)

lower = min(
    y_test.min(),
    rf_lite_test_pred.min(),
)
upper = max(
    y_test.max(),
    rf_lite_test_pred.max(),
)

plt.plot(
    [lower, upper],
    [lower, upper],
    linestyle="--",
)

plt.xlabel("Derived EFI")
plt.ylabel("Predicted EFI")
plt.title(
    "Random Forest Lite: Predicted vs Derived EFI"
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "rf_lite_predicted_vs_derived.png",
    bbox_inches="tight",
)
plt.show()

In [ ]:
rf_lite_residuals = (
    y_test - rf_lite_test_pred
)

plt.figure(figsize=(7, 5))
plt.scatter(
    rf_lite_test_pred,
    rf_lite_residuals,
    alpha=0.4,
)
plt.axhline(0, linestyle="--")

plt.xlabel("Predicted EFI")
plt.ylabel(
    "Residual (derived - predicted)"
)
plt.title(
    "Random Forest Lite Residuals"
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "rf_lite_residuals.png",
    bbox_inches="tight",
)
plt.show()

In [ ]:
train_sizes, train_scores, validation_scores = learning_curve(
    estimator=rf_lite,
    X=X_raw,
    y=y,
    cv=3,
    scoring="r2",
    train_sizes=np.linspace(
        0.1,
        1.0,
        10,
    ),
    shuffle=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

plt.figure(figsize=(8, 5))
plt.plot(
    train_sizes,
    train_scores.mean(axis=1),
    marker="o",
    label="Training score",
)
plt.plot(
    train_sizes,
    validation_scores.mean(axis=1),
    marker="o",
    label="Validation score",
)

plt.xlabel("Training set size")
plt.ylabel("R²")
plt.title(
    "Learning Curve — Random Forest Lite"
)
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "rf_lite_learning_curve.png",
    bbox_inches="tight",
)
plt.show()

## 10. Explainability

Global feature importance and SHAP are used to examine how the Random Forest Lite surrogate model represents the relationship between the 12 emission measurements and the derived EFI.

A reproducible sample of at most 1000 erservered development observations is used for SHAP visualization.


In [ ]:
feature_importance_df = (
    pd.DataFrame({
        "Feature": FEATURE_COLUMNS,
        "Importance": rf_lite.feature_importances_,
    })
    .sort_values(
        "Importance",
        ascending=False,
    )
)

feature_importance_df


In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(
    data=feature_importance_df,
    x="Importance",
    y="Feature",
)

plt.title(
    "Random Forest Lite Feature Importance"
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "rf_lite_feature_importance.png",
    bbox_inches="tight",
)
plt.show()

In [ ]:
X_test_df = pd.DataFrame(
    X_test_raw,
    columns=FEATURE_COLUMNS,
)

shap_sample = X_test_df.sample(
    n=min(1000, len(X_test_df)),
    random_state=RANDOM_STATE,
)

explainer = shap.TreeExplainer(
    rf_lite
)
shap_values = explainer.shap_values(
    shap_sample
)

shap.summary_plot(
    shap_values,
    shap_sample,
    plot_type="bar",
    show=False,
)
plt.title(
    "SHAP Global Feature Importance — Random Forest Lite"
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "shap_global_importance.png",
    bbox_inches="tight",
)
plt.show()

In [ ]:
shap.summary_plot(
    shap_values,
    shap_sample,
    show=False,
)

plt.title(
    "SHAP Summary — Random Forest Lite"
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "shap_summary.png",
    bbox_inches="tight",
)
plt.show()

## 11. Temporal evaluation on March data

March is kept outside the January–February EFI-development workflow. The trained Random Forest Lite model is applied directly to the preprocessed March emission measurements to generate `Predicted_EFI`.

This section examines prediction behaviour and distributional consistency on the preserved month. March does not provide an independently measured EFI target. Therefore, the analysis is interpreted as temporal prediction/distribution evaluation rather than direct validation against EFI ground truth.


In [ ]:
march_df["Predicted_EFI"] = rf_lite.predict(
    march_df[FEATURE_COLUMNS].to_numpy()
)

print(
    march_df["Predicted_EFI"].describe()
)

In [ ]:
march_negative_count = int(
    (march_df["Predicted_EFI"] < 0).sum()
)
march_negative_pct = (
    100
    * march_negative_count
    / len(march_df)
)

print(
    "March Predicted_EFI < 0 records: "
    f"{march_negative_count:,}"
)
print(
    "March Predicted_EFI < 0 percentage: "
    f"{march_negative_pct:.2f}%"
)

### 11.1 Predicted-EFI grouping in March

One dimensional K-means clustering is applied to the March predicted EFI scores as a descriptive analysis of whether the predictions form three score groups. These groups are not treated as independently observed engine-condition labels.

In [ ]:
march_efi_kmeans = KMeans(
    n_clusters=3,
    random_state=RANDOM_STATE,
    n_init=10,
)

march_df["Predicted_EFI_Cluster"] = (
    march_efi_kmeans.fit_predict(
        march_df[["Predicted_EFI"]]
    )
)

march_cluster_summary = (
    march_df
    .groupby(
        "Predicted_EFI_Cluster"
    )["Predicted_EFI"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max",
        ]
    )
    .sort_values("mean")
)

march_cluster_summary

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=march_df,
    x="Predicted_EFI_Cluster",
    y="Predicted_EFI",
)

plt.title(
    "March Predicted EFI by Descriptive K-means Group"
)
plt.xlabel("Predicted-EFI cluster")
plt.ylabel("Predicted EFI")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "march_predicted_efi_clusters.png",
    bbox_inches="tight",
)
plt.show()

### 11.2 Development EFI vs March predicted EFI

The January–February derived EFI distribution and March predicted-EFI distribution are compared using descriptive statistics, density plots, and a two-sample Kolmogorov–Smirnov (KS) test.

The KS test evaluates equality of the two empirical distributions. A statistically significant result indicates a distributional difference and is not interpreted by itself as a measure of predictive accuracy.


In [ ]:
distribution_summary = pd.DataFrame({
    "Development EFI":
        development_df["EFI_Score"].describe(),
    "March Predicted EFI":
        march_df["Predicted_EFI"].describe(),
})

distribution_summary


In [ ]:
ks_statistic, ks_p_value = ks_2samp(
    development_df["EFI_Score"],
    march_df["Predicted_EFI"],
)

print(
    f"KS statistic: {ks_statistic:.6f}"
)
print(
    f"p-value: {ks_p_value:.6g}"
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.kdeplot(
    development_df["EFI_Score"],
    label="Development EFI",
    fill=True,
    alpha=0.35,
)
sns.kdeplot(
    march_df["Predicted_EFI"],
    label="March Predicted EFI",
    fill=True,
    alpha=0.35,
)

plt.title(
    "EFI Distribution: Development vs March Prediction"
)
plt.xlabel("EFI")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "development_vs_march_efi.png",
    bbox_inches="tight",
)
plt.show()

## 12. Methodological summary

The EFI workflow combines unsupervised profile discovery, an empirically calibrated distance-based index, supervised learning modelling, explainability analysis, and temporally preserved evaluation.

The key methodological sequence is:

- Twelve available petrol-vehicle emission/operating measurements are standardized.
- K-means with $k=3$ identifies three broad multivariate emission profiles,
- Cluster 2 is selected as the empirical clean reference from its observed emission characteristics,
- Euclidean distance from the clean-cluster centroid quantifies multivariate deviation from that reference,
- the distance reference is empirically calibrated to preserve the small extreme tail observed in the development distribution,
- the scaled distance is inverted so that higher EFI represents greater similarity to the clean reference,
- multiple regression models are evaluated as surrogate predictors of the derived EFI,
- Random Forest Lite is interpreted using feature importance and SHAP, and
- March records are held out for temporal prediction and distribution analysis.
